# 50 — Instrument Category-Token Resolution

Resolves the COSMoS `bc_categories` tokens used by instrument Dataset Specializations
(domains QS, FT, RS) to a Biomedical Concept identity, where one exists.

**Why.** `BC_Categories` is the working group's confirmed instrument-family grouping
mechanism, but the tokens it carries are labels, not addressable identifiers. Some labels
coincide with a `bc_short_name` and resolve to a BC ID; most do not. This is the
identifier asymmetry described in `docs/COSMoS_Instrument_Layer.md` §5. The `bc_synonyms`
field is the machine-readable link the asymmetry leaves implicit — recovering it is the
job of this notebook.

**Design rule.** Resolution ships as a status enum
(`exact_name | synonym | synonym_ambiguous | unresolved`), never a bare
`category -> bc_id` column. Bare nulls would misrepresent by-design category buckets
(e.g. "QRS", "Questionnaires", "RECIST 1.1") as broken links. Deterministic only —
exact match then unique-synonym match, no fuzzy matching, no fabrication.

**Version caveat.** Counts are package snapshots and re-derive on every BC/DSS update.
Nothing version-bound is hardcoded here. Use the optional diff cell at the end to compare
a prior run against the current one on each package bump.

**Inputs / outputs.**
- Reads `interim/COSMoS_Graph.xlsx` (sheets `BC`, `DSS`, `BC_Categories`) — runs after the
  flatten (`10_flatten_schema_driven.ipynb`).
- Writes `reports/Instrument_Category_Resolution.xlsx`.


In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

BASE_DIR = Path.cwd().parent          # cosmos-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
GRAPH_FILE = BASE_DIR / 'interim' / 'COSMoS_Graph.xlsx'
REPORTS_DIR = BASE_DIR / 'reports'
OUTPUT_FILE = REPORTS_DIR / 'Instrument_Category_Resolution.xlsx'

# Instrument structural types (sdtm-findings-graph instrument sub-type).
INSTR_DOMAINS = {'QS', 'FT', 'RS'}

# Optional: path to a previous Instrument_Category_Resolution.xlsx for the version-bump
# diff in the final cell. Leave as None to skip.
PRIOR_FILE = None

print('Graph: ', GRAPH_FILE.relative_to(REPO_ROOT))
print('Output:', OUTPUT_FILE.relative_to(REPO_ROOT))


## Token population

Instrument-scope tokens are the distinct `BC_Categories.category` values carried by BCs
that have at least one Dataset Specialization in an instrument domain (QS, FT, RS). This
is the population that reproduces the `COSMoS_Instrument_Layer.md` §5 anchor.


In [ ]:
bc  = pd.read_excel(GRAPH_FILE, sheet_name='BC')
dss = pd.read_excel(GRAPH_FILE, sheet_name='DSS')
bcc = pd.read_excel(GRAPH_FILE, sheet_name='BC_Categories')

instrument_bc_ids = set(dss.loc[dss['domain'].isin(INSTR_DOMAINS), 'bc_id'])
tokens = sorted(set(bcc.loc[bcc['bc_id'].isin(instrument_bc_ids), 'category'].dropna()))

print(f'Instrument BCs (DSS in {sorted(INSTR_DOMAINS)}): {len(instrument_bc_ids)}')
print(f'Instrument-scope category tokens:        {len(tokens)}')


## Resolution mechanism

Two deterministic passes per token:

1. **`exact_name`** — token equals a `bc_short_name` (case-sensitive, the form COSMoS
   publishes).
2. **`synonym`** — token matches exactly one BC via `bc_synonyms` (lower-cased, `;`-split).
   Matches against more than one BC are **`synonym_ambiguous`**.

Anything unmatched is **`unresolved`** — by-design buckets, classification frameworks, and
genuine abbreviation gaps, left undistinguished (classifying them is QC of CDISC content,
out of scope here).


In [ ]:
# Pass 1 lookup: exact bc_short_name -> [bc_id, ...]
shortname_to_bc = {}
for _, r in bc.iterrows():
    shortname_to_bc.setdefault(str(r['bc_short_name']).strip(), []).append(r['bc_id'])

# Pass 2 lookup: synonym (lower-cased) -> {bc_id, ...}
synonym_to_bc = {}
for _, r in bc.iterrows():
    if pd.isna(r['bc_synonyms']):
        continue
    for syn in str(r['bc_synonyms']).split(';'):
        syn = syn.strip()
        if syn:
            synonym_to_bc.setdefault(syn.lower(), set()).add(r['bc_id'])

def resolve(token):
    if token in shortname_to_bc:
        return 'exact_name', shortname_to_bc[token]
    hits = synonym_to_bc.get(token.lower())
    if hits:
        if len(hits) == 1:
            return 'synonym', list(hits)
        return 'synonym_ambiguous', sorted(hits)
    return 'unresolved', []

shortname_by_bc = bc.set_index('bc_id')['bc_short_name'].to_dict()

records = []
for token in tokens:
    status, bc_ids = resolve(token)
    records.append({
        'category': token,
        'status': status,
        'bc_id': ';'.join(map(str, bc_ids)),
        'bc_short_name': ';'.join(str(shortname_by_bc.get(b, '')) for b in bc_ids),
    })

resolution = pd.DataFrame(records, columns=['category', 'status', 'bc_id', 'bc_short_name'])
resolution.head(12)


## Summary


In [ ]:
counts = Counter(resolution['status'])
resolved = counts['exact_name'] + counts['synonym']
summary = pd.DataFrame(
    [(s, counts.get(s, 0)) for s in ['exact_name', 'synonym', 'synonym_ambiguous', 'unresolved']],
    columns=['status', 'token_count'],
)
print(summary.to_string(index=False))
print(f'\nResolved (exact_name + synonym): {resolved} / {len(resolution)}')


## Write report

`reports/Instrument_Category_Resolution.xlsx` — README, Resolution, Summary sheets.
Header colours follow the repo convention: yellow (`FFD700`) for COSMoS-layer columns,
grey (`808080`) for key columns.


In [ ]:
REPORTS_DIR.mkdir(exist_ok=True)

YELLOW = PatternFill('solid', fgColor='FFD700')   # COSMoS layer
GREY   = PatternFill('solid', fgColor='808080')   # keys
HDR_BLACK = Font(bold=True, color='000000')
HDR_WHITE = Font(bold=True, color='FFFFFF')
BODY = Font(size=11)

wb = Workbook()

# --- README ---
ws = wb.active
ws.title = 'README'
readme = [
    ('Instrument Category-Token Resolution', Font(bold=True, size=14)),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', BODY),
    ('', BODY),
    ('Resolves instrument-scope COSMoS bc_categories tokens to BC identity.', BODY),
    ('Source: cosmos-graph/interim/COSMoS_Graph.xlsx (BC, DSS, BC_Categories).', BODY),
    ('Population: distinct BC_Categories tokens on BCs with a DSS in QS/FT/RS.', BODY),
    ('', BODY),
    ('Resolution sheet columns:', Font(bold=True, size=11)),
    ('  category       - the BC_Categories token (key).', BODY),
    ('  status         - exact_name | synonym | synonym_ambiguous | unresolved.', BODY),
    ('  bc_id          - resolved BC id(s), ; -separated (key).', BODY),
    ('  bc_short_name  - resolved BC short name(s).', BODY),
    ('', BODY),
    ('Mechanism: exact bc_short_name, then unique bc_synonyms (lower-cased, ;-split).', BODY),
    ('Deterministic only - no fuzzy matching, no fabrication.', BODY),
    ('Design rule: status enum, never a bare category->bc_id column. unresolved', BODY),
    ('  includes by-design buckets (QRS, Questionnaires) and frameworks (RECIST 1.1).', BODY),
    ('', BODY),
    ('Counts are package-version snapshots; re-run and diff on each BC/DSS bump.', BODY),
]
for i, (text, font) in enumerate(readme, start=1):
    c = ws.cell(row=i, column=1, value=text)
    c.font = font
ws.column_dimensions['A'].width = 92

def write_sheet(df, title, key_cols):
    ws = wb.create_sheet(title)
    for j, col in enumerate(df.columns, start=1):
        c = ws.cell(row=1, column=j, value=col)
        if col in key_cols:
            c.fill = GREY; c.font = HDR_WHITE
        else:
            c.fill = YELLOW; c.font = HDR_BLACK
        c.alignment = Alignment(horizontal='left')
    for i, (_, row) in enumerate(df.iterrows(), start=2):
        for j, col in enumerate(df.columns, start=1):
            ws.cell(row=i, column=j, value=row[col])
    for j, col in enumerate(df.columns, start=1):
        values = [len(str(col))] + [len(str(v)) for v in df[col]]
        ws.column_dimensions[get_column_letter(j)].width = min(max(values) + 2, 60)
    ws.freeze_panes = 'A2'

write_sheet(resolution, 'Resolution', key_cols={'category', 'bc_id'})
write_sheet(summary, 'Summary', key_cols={'status'})

wb.save(OUTPUT_FILE)
print('Wrote', OUTPUT_FILE.relative_to(REPO_ROOT))


## Version-bump diff (optional)

Set `PRIOR_FILE` (cell 2) to a previous `Instrument_Category_Resolution.xlsx` to compare
status transitions against the current package. No-ops if unset or missing.


In [ ]:
if PRIOR_FILE and Path(PRIOR_FILE).exists():
    prior = pd.read_excel(PRIOR_FILE, sheet_name='Resolution').fillna('')
    prev = dict(zip(prior['category'], prior['status']))
    curr = dict(zip(resolution['category'], resolution['status']))
    added = sorted(set(curr) - set(prev))
    dropped = sorted(set(prev) - set(curr))
    changed = [(t, prev[t], curr[t]) for t in sorted(set(prev) & set(curr)) if prev[t] != curr[t]]
    print(f'Prior:   {PRIOR_FILE}')
    print(f'Tokens:  prior {len(prev)} -> current {len(curr)}')
    print(f'Added tokens:   {added}')
    print(f'Dropped tokens: {dropped}')
    print('Status changes (token: prior -> current):')
    for t, a, b in changed:
        print(f'  {t}: {a} -> {b}')
    if not changed:
        print('  (none)')
else:
    print('PRIOR_FILE not set or missing - skipping version diff.')
